# Proyecto de Analítica de Datos: Coffee Shop Sales
**Metodología:** CRISP-DM (Fase 1: Entendimiento del Negocio y Fase 2: Entendimiento de los Datos)

---

## 1. Entendimiento del Negocio (Business Understanding)

### 1.1 Contexto de la Empresa
La cadena de cafeterías opera en ubicaciones clave con un flujo constante de clientes diarios. Para mantener la competitividad y maximizar la rentabilidad, la administración requiere comprender los patrones de compra, el desempeño por sucursal, la demanda por categoría de producto y las horas de mayor afluencia.

### 1.2 Objetivos del Proyecto
1. **Analizar el Desempeño Operativo y Comercial:** Identificar el volumen de ventas e ingresos generados por sucursal (`store_location`) y detectar variaciones temporales.
2. **Optimización de Inventario y Menú:** Determinar los productos y categorías de mayor y menor rotación para ajustar compras de insumos y promociones.
3. **Identificación de Patrones de Consumo:** Analizar la distribución de transacciones a lo largo del día y de la semana para optimizar la programación del personal en barra.
4. **Estructuración y Calidad de Datos:** Formalizar un diccionario de datos y un modelo relacional normalizado que permita escalar la infraestructura analítica del negocio.

### 1.3 Criterios de Éxito del Negocio
- Generación de insights accionables para la toma de decisiones comerciales e inventario.
- Identificación clara de horas pico y productos estrella por ubicación.

## 2. Entendimiento de los Datos (Data Understanding)

### 2.1 Descripción de los Datos Disponibles
El conjunto de datos proviene del archivo de registro transaccional `Coffee Shop Sales.xlsx`. Contiene **149,116 registros** correspondientes al primer semestre del año 2023 (del 1 de enero al 30 de junio de 2023). Cada registro representa una línea de detalle dentro de una transacción efectuada en alguna de las sucursales de la cadena.

### 2.2 Diccionario de Datos

| Campo | Tipo de Dato | Nulo | Descripción | Rango / Ejemplo |
| :--- | :--- | :---: | :--- | :--- |
| **`transaction_id`** | `INTEGER` | No | Identificador único de cada línea de transacción (Clave Primaria). | `1` a `149,116` |
| **`transaction_date`** | `DATE` | No | Fecha en que se realizó la venta (AAAA-MM-DD). | `2023-01-01` a `2023-06-30` |
| **`transaction_time`** | `TIME` | No | Hora exacta de la transacción (HH:MM:SS). | `07:06:11` |
| **`transaction_qty`** | `INTEGER` | No | Cantidad de unidades vendidas del producto. | Min: `1`, Max: `8` |
| **`store_id`** | `INTEGER` | No | Identificador único de la sucursal. | `3` (Astoria), `5` (Lower Manhattan), `8` (Hell's Kitchen) |
| **`store_location`** | `VARCHAR(50)` | No | Nombre o ubicación geográfica de la sucursal. | `Lower Manhattan`, `Hell's Kitchen`, `Astoria` |
| **`product_id`** | `INTEGER` | No | Identificador único del producto. | `1` a `80` |
| **`unit_price`** | `DECIMAL(10,2)` | No | Precio unitario del producto en moneda local. | Min: `$0.80`, Max: `$45.00` |
| **`product_category`**| `VARCHAR(50)` | No | Categoría general del producto (9 categorías). | `Coffee`, `Tea`, `Drinking Chocolate`, `Bakery`, etc. |
| **`product_type`** | `VARCHAR(50)` | No | Subcategoría o variedad del producto (29 tipos). | `Gourmet brewed coffee`, `Brewed Chai tea` |
| **`product_detail`** | `VARCHAR(100)`| No | Presentación específica y tamaño del producto. | `Ethiopia Rg`, `Spicy Eye Opener Chai Lg` |

### 2.3 Arquitectura y Diagrama de Datos (Modelo Relacional)

La base de datos original se presenta desnormalizada en una sola tabla. A continuación se define la propuesta del modelo entidad-relación normalizado (Esquema en Estrella):

```mermaid
erdiagram
    STORES ||--o{ TRANSACTIONS : "registra"
    PRODUCTS ||--o{ TRANSACTIONS : "es vendido en"

    STORES {
        int store_id PK
        string store_location
    }

    PRODUCTS {
        int product_id PK
        string product_category
        string product_type
        string product_detail
        decimal unit_price
    }

    TRANSACTIONS {
        int transaction_id PK
        date transaction_date
        time transaction_time
        int transaction_qty
        int store_id FK
        int product_id FK
    }
```

---
## 3. Acceso a Datos y Análisis Descriptivo en Google Colab

In [ ]:
# Setup e Importación de Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo gráfico
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

In [ ]:
# Instrucciones para Carga de Datos en Colab
'''
Si estás ejecutando este archivo directamente en Google Colab, ejecuta las siguientes líneas para subir el archivo 'Coffee Shop Sales.xlsx':

from google.colab import files
uploaded = files.upload()
'''

file_path = 'Coffee Shop Sales.xlsx'
df = pd.read_excel(file_path)

# Cálculo de la columna derivada Total_Sales ($)
df['total_sales'] = df['transaction_qty'] * df['unit_price']

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

In [ ]:
# Inspección General y Tipos de Datos
print("=== INFORMACIÓN GENERAL ===")
df.info()

print("\n=== VALORES NULOS POR COLUMNA ===")
print(df.isnull().sum())

print("\n=== VALORES ÚNICOS POR COLUMNA ===")
print(df.nunique())

In [ ]:
# Resumen Estadístico de Variables Numéricas
df[['transaction_qty', 'unit_price', 'total_sales']].describe().T

In [ ]:
# 3.1 Desempeño Comercial por Sucursal
store_summary = df.groupby('store_location').agg(
    total_transactions=('transaction_id', 'count'),
    total_units_sold=('transaction_qty', 'sum'),
    total_revenue=('total_sales', 'sum'),
    avg_ticket=('total_sales', 'mean')
).reset_index()

print("=== RESUMEN POR SUCURSAL ===")
display(store_summary)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=store_summary, x='store_location', y='total_revenue', palette='viridis', ax=ax[0])
ax[0].set_title('Ingresos Totales por Sucursal ($)')
ax[0].set_ylabel('Ingresos ($)')
ax[0].set_xlabel('Sucursal')

sns.barplot(data=store_summary, x='store_location', y='total_transactions', palette='magma', ax=ax[1])
ax[1].set_title('Total de Transacciones por Sucursal')
ax[1].set_ylabel('N° Transacciones')
ax[1].set_xlabel('Sucursal')
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Venta por Categoría de Producto
cat_summary = df.groupby('product_category').agg(
    units_sold=('transaction_qty', 'sum'),
    revenue=('total_sales', 'sum')
).sort_values(by='revenue', ascending=False).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=cat_summary, x='revenue', y='product_category', palette='rocket')
plt.title('Ingresos Generados por Categoría de Producto')
plt.xlabel('Ingresos Totales ($)')
plt.ylabel('Categoría')
plt.show()

cat_summary

In [ ]:
# 3.3 Tendencia Mensual e Identificación de Horas Pico
df['month'] = df['transaction_date'].dt.strftime('%Y-%m (%B)')
df['hour'] = pd.to_datetime(df['transaction_time'].astype(str)).dt.hour

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Tendencia mensual
monthly_sales = df.groupby('month')['total_sales'].sum().reset_index()
sns.lineplot(data=monthly_sales, x='month', y='total_sales', marker='o', color='crimson', linewidth=2.5, ax=ax[0])
ax[0].set_title('Evolución Mensual de Ingresos (Enero - Junio 2023)')
ax[0].set_ylabel('Ingresos Totales ($)')
ax[0].set_xlabel('Mes')
ax[0].tick_params(axis='x', rotation=30)

# Distribución horaria
hourly_sales = df.groupby('hour')['transaction_id'].count().reset_index()
sns.barplot(data=hourly_sales, x='hour', y='transaction_id', palette='Blues_d', ax=ax[1])
ax[1].set_title('Volumen de Transacciones por Hora del Día (Horas Pico)')
ax[1].set_ylabel('Cantidad de Transacciones')
ax[1].set_xlabel('Hora del Día (Formato 24h)')

plt.tight_layout()
plt.show()

## 4. Conclusiones y Hallazgos Principales

1. **Volumen e Ingresos por Sucursal:** Las sucursales presentan un volumen de ventas muy equilibrado con una leve ventaja en Hell's Kitchen y Lower Manhattan.
2. **Categorías Dominantes:** Las categorías de `Coffee`, `Tea` y `Drinking Chocolate` concentran la mayor porción de los ingresos operativos del negocio.
3. **Horas Pico:** Se evidencia una clara concentración del flujo de transacciones en el horario matutino (entre las 7:00 AM y las 10:00 AM), disminuyendo gradualmente hacia la tarde.
4. **Tendencia Creciente:** Existe un incremento sostenido en las ventas mes a mes desde enero hasta junio de 2023.